In [ ]:
import os
import json
import joblib
import pandas as pd

os.chdir(r"C:\Users\Pelin\Desktop\Yeni klasör")
print("Şu anki dizin:", os.getcwd())

MODEL_PATH = "xgb_final.pkl"
CONF_PATH  = "bug_predict_config.json"

xgb_model = joblib.load(MODEL_PATH)               #Loading the model and configuration

with open(CONF_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FEATURES = cfg["features"]
BEST_THR = cfg["best_threshold"]

print("Model ve config yüklendi ✅")
print("Özellik sayısı:", len(FEATURES))
print("Eşik (best_threshold):", BEST_THR)


Şu anki dizin: C:\Users\Pelin\Desktop\Yeni klasör
Model ve config yüklendi ✅
Özellik sayısı: 21
Eşik (best_threshold): 0.529


In [ ]:
import ast, io, tokenize, math, re
import numpy as np
import pandas as pd

_PY_KEYWORDS = {
    'False','None','True','and','as','assert','async','await','break','class','continue',
    'def','del','elif','else','except','finally','for','from','global','if','import','in',
    'is','lambda','nonlocal','not','or','pass','raise','return','try','while','with','yield'
}
_OP_TOKENS = {
    '(',')','[',']','{','}','.',',',':',';','+','-','*','**','/','//','%','@',
    '<<','>>','&','|','^','~','<','>','<=','>=','==','!='
}

def _count_lines(code:str):
    lines = code.splitlines()
    l_blank = sum(1 for l in lines if l.strip()=='')
    l_comment = 0
    try:
        for tok in tokenize.generate_tokens(io.StringIO(code).readline):
            if tok.type == tokenize.COMMENT:
                l_comment += 1
    except tokenize.TokenError:
        pass
    
    l_total = len(lines)
    l_code = max(l_total - l_blank - l_comment, 0)
    return l_total, l_code, l_comment, l_blank

def _branch_and_cyclomatic(tree: ast.AST):
    branch = 0
    for node in ast.walk(tree):
        if isinstance(node, (ast.If, ast.For, ast.While, ast.With, ast.AsyncWith)):
            branch += 1
        elif isinstance(node, ast.Try):
            branch += max(1, len(getattr(node, 'handlers', [])))
        elif isinstance(node, ast.BoolOp):
            if isinstance(node.op, (ast.And, ast.Or)):
                branch += max(1, len(node.values)-1)
        elif isinstance(node, (ast.IfExp,)):  
            branch += 1
        elif isinstance(node, (ast.comprehension,)):
            branch += 1
    vg = 1 + branch
    return branch, vg

def _halstead(code:str):
    uniq_ops=set(); uniq_opnds=set(); tot_ops=0; tot_opnds=0
    try:
        for tok in tokenize.generate_tokens(io.StringIO(code).readline):
            ttype, tstr = tok.type, tok.string
            if ttype == tokenize.OP and tstr.strip():
                uniq_ops.add(tstr); tot_ops += 1
            elif ttype == tokenize.NAME:
                if tstr in _PY_KEYWORDS:
                    uniq_ops.add(tstr); tot_ops += 1
                else:
                    uniq_opnds.add(tstr); tot_opnds += 1
            elif ttype in (tokenize.NUMBER, tokenize.STRING):
                uniq_opnds.add(tstr); tot_opnds += 1
    except tokenize.TokenError:
        pass

    n1 = len(uniq_ops);    N1 = tot_ops
    n2 = len(uniq_opnds);  N2 = tot_opnds
    n  = n1 + n2
    N  = N1 + N2
    v = N * math.log2(n) if n>0 else 0.0
    d = ((n1/2.0) * (N2/max(n2,1))) if n2>0 else 0.0
    l = (1.0/d) if d>0 else 0.0
    i = l * v                 
    e = d * v                  
    t = e / 18.0               
    b = v / 3000.0             
    
    return {
        "uniq_Op": n1, "uniq_Opnd": n2, "total_Op": N1, "total_Opnd": N2,
        "n": n, "v": v, "l": l, "d": d, "i": i, "e": e, "t": t, "b": b
    }

def extract_metrics_from_code(code: str) -> dict:
    loc_total, l_code, l_comment, l_blank = _count_lines(code)

    try:
        tree = ast.parse(code)
        branch, vg = _branch_and_cyclomatic(tree)
    except SyntaxError:
        branch, vg = 0, 1

    hs = _halstead(code)

    metrics = dict(
        loc=float(l_code),
        **{"v(g)": float(vg)},
        **{"ev(g)": float(vg)*0.60, "iv(g)": float(vg)*0.40},  # proxy
        n=float(hs["n"]),
        v=float(hs["v"]),
        l=float(hs["l"]),
        d=float(hs["d"]),
        i=float(hs["i"]),
        e=float(hs["e"]),
        b=float(hs["b"]),
        t=float(hs["t"]),
        lOCode=int(l_code),
        lOComment=int(l_comment),
        lOBlank=int(l_blank),
        locCodeAndComment=int(l_code + l_comment),
        uniq_Op=int(hs["uniq_Op"]),
        uniq_Opnd=int(hs["uniq_Opnd"]),
        total_Op=int(hs["total_Op"]),
        total_Opnd=int(hs["total_Opnd"]),
        branchCount=int(branch),
    )

    for k,vv in list(metrics.items()):
        if isinstance(vv, (int,float)):
            if np.isnan(vv) or np.isinf(vv): metrics[k]=0.0
            if k in {"lOCode","lOComment","lOBlank","locCodeAndComment","uniq_Op","uniq_Opnd","total_Op","total_Opnd","branchCount"}:
                metrics[k]=int(max(0, metrics[k]))
            if k=="loc":
                metrics[k]=float(max(0, metrics[k]))

    metrics["total_Op"]  = max(metrics["total_Op"],  metrics["uniq_Op"])
    metrics["total_Opnd"]= max(metrics["total_Opnd"],metrics["uniq_Opnd"])
    return metrics


In [ ]:

def build_row_for_model(m: dict, feature_order):

    row = {f: m.get(f, 0) for f in feature_order}
    return pd.DataFrame([row], columns=feature_order)

def predict_defect_from_code(code_str: str, verbose=True):
    m = extract_metrics_from_code(code_str)
    X = build_row_for_model(m, FEATURES)
    proba = float(xgb_model.predict_proba(X)[:,1][0])
    label = int(proba >= BEST_THR)
    if verbose:
        print("— Extracted metrics (subset) —")
        show = ["loc","lOCode","lOComment","lOBlank","locCodeAndComment","branchCount","v(g)","uniq_Op","uniq_Opnd","total_Op","total_Opnd","v","d","e","b"]
        for k in show:
            print(f"{k:>18} : {m[k]}")
        print(f"\nPredicted P(defect=1) = {proba:.3f} | threshold={BEST_THR:.3f} → label={label}")
    return proba, label, m


In [18]:
code_snippet = r"""
import random, string

class Creature:
    def __init__(self, dna=None):
        self.dna = dna or ''.join(random.choices("ATCG", k=8))
        self.fitness = self.dna.count('A') + random.random()

    def mutate(self):
        self.dna = ''.join(
            c if random.random() > 0.2 else random.choice("ATCG")
            for c in self.dna
        )
        self.fitness = self.dna.count('A') + random.random()

def evolve(pop=5, gen=5):
    creatures = [Creature() for _ in range(pop)]
    for g in range(gen):
        for c in creatures: c.mutate()
        creatures.sort(key=lambda x: x.fitness, reverse=True)
        print(f"\nGen {g+1}: {[c.dna for c in creatures]}")
        best = creatures[:2]
        creatures = best + [Creature(random.choice(best).dna) for _ in range(pop - 2)]

evolve()
"""

proba, label, metrics = predict_defect_from_code(code_snippet, verbose=True)


— Extracted metrics (subset) —
               loc : 20.0
            lOCode : 20
         lOComment : 0
           lOBlank : 5
 locCodeAndComment : 20
       branchCount : 8
              v(g) : 9.0
           uniq_Op : 24
         uniq_Opnd : 35
          total_Op : 133
        total_Opnd : 94
                 v : 1335.359972205138
                 d : 32.22857142857143
                 e : 43036.74424706845
                 b : 0.44511999073504604

Predicted P(defect=1) = 0.137 | threshold=0.529 → label=0


In [ ]:
# High risk example
code_risky = r"""
import os, json, random
STATE = {"cache": {}, "tries": 0}

def parse_csv(path, sep=",", cache=STATE.get("cache", {})):   
  
    if not isinstance(path, str):
        raise ValueError("path must be str")
    if path in cache:                    
        return cache[path]
    rows = []
    try:
        f = open(path, "r")                
        for line in f:
            if not line.strip():
                continue
            if sep in line:
                rows.append(line.strip().split(sep))
            elif ";" in line:
                rows.append(line.strip().split(";"))
            else:
                
                try:
                    rows.append(eval(line.strip()))          
                except Exception:
                    pass                                    
       
        avg_len = sum(len(r) for r in rows) / len(rows)      
        if avg_len > 10 and random.random() > 0.5:
            raise RuntimeError("suspicious width")
        cache[path] = rows
        return rows
    except FileNotFoundError:
        return []
    except Exception as e:
        if "suspicious" in str(e).lower():
            return rows[:1]
        elif STATE["tries"] < 2:
            STATE["tries"] += 1
            return parse_csv(path, sep=sep)                  
        else:
            return []
    finally:
        try:
            f.close()
        except Exception:
            pass

def compute_scores(items, weights=None):
  
    weights = weights or [random.randint(0, 2) for _ in range(len(items))]
    s = 0; w = 0
    for i, it in enumerate(items):
        if isinstance(it, (list, tuple)):
            s += len(it) * (weights[i] if i < len(weights) else 1)
            w += (weights[i] if i < len(weights) else 1)
        elif isinstance(it, dict):
            s += len(it.keys())
            w += 1
        else:
            try:
                s += float(it)
                w += 1
            except Exception:
                if it is None:
                    continue
                elif it == "":
                    break
                else:
                    w += 0
    if w == 0 and random.random() < 0.5:
        w = random.choice([0, 1])                          
    return s / w                                             

def main(path):
    data = parse_csv(path, sep=",")
    total = 0
    i = 0
    while i < len(data) + 3:                                 
        try:
            row = data[i] if i < len(data) else []
            score = compute_scores(row, weights=[1,0,1,0,1][:len(row)])
            if score > 5:
                total += score
            elif score == 0:
                total += 1
            else:
                total -= 0.5
        except Exception:
          
            pass
        i += 1
    return {"total": total, "n": len(data), "ok": total > 10}

if __name__ == "__main__":
    print(main("maybe_missing_file.csv"))
"""

proba, label, metrics = predict_defect_from_code(code_risky, verbose=True)


— Extracted metrics (subset) —
               loc : 76.0
            lOCode : 76
         lOComment : 15
           lOBlank : 5
 locCodeAndComment : 91
       branchCount : 33
              v(g) : 34.0
           uniq_Op : 41
         uniq_Opnd : 75
          total_Op : 361
        total_Opnd : 227
                 v : 4032.4928251350125
                 d : 62.04666666666667
                 e : 250202.73815687708
                 b : 1.3441642750450042

Predicted P(defect=1) = 0.642 | threshold=0.529 → label=1
